# MIDI Preprocessing — CodeAlpha Task 3
**Music Generation with LSTM**

This notebook:
1. Loads MIDI files from Maestro v3 + GiantMIDI-Piano
2. Extracts **(pitch, duration, velocity)** tuples using `symusic` (C++ parser, 10-100x faster than music21)
3. Discretizes durations (10 musical buckets) and velocities (8 nuance buckets: ppp → fff)
4. Encodes sequences as integers
5. Saves `X.npy`, `y.npy`, `vocab.json` for training

**Key features:**
- Runs on **Kaggle** (auto-detected) or **local** — no config change needed
- **Stream to disk** — events written chunk by chunk, RAM stays flat
- **Checkpoint & resume** — saves every 500 files, re-run to continue
- **music21** kept for MIDI conversion in notebook 2 (generation)

## 1. Install dependencies

In [1]:
!pip install symusic numpy tqdm -q

## 2. Imports & configuration

In [2]:
import os
import json
import glob
import pickle
import logging
import time
import sys
from collections import Counter

import numpy as np
import symusic
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)

# ── Environment detection ─────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/input/')
logger.info("Environment: %s", "KAGGLE" if ON_KAGGLE else "LOCAL")

if ON_KAGGLE:
    MAESTRO_DIR   = "/kaggle/input/datasets/kritanjalijain/maestropianomidi/maestro-v3.0.0"
    GIANTMIDI_DIR = "/kaggle/input/datasets/pictureinthenoise/music-generation-with-giantmidi-piano/giantmidi-piano-unzipped-midi-v1.21-clean"
    OUTPUT_DIR    = "/kaggle/working"
else:
    MAESTRO_DIR   = "/home/borrelle/Working/codealpha_project/task3-midi-musique-generator/datasets/maestro-v3.0.0"
    GIANTMIDI_DIR = "/home/borrelle/Working/codealpha_project/task3-midi-musique-generator/datasets/giantmidi-piano-unzipped-midi-v1.21-clean"
    OUTPUT_DIR    = "./output"

# ── Config ────────────────────────────────────────────────────────────────────
SEQUENCE_LEN     = 64
MIN_NOTES        = 50
MAX_FILES        = None     # None = all; set e.g. 10 for a quick test
CHECKPOINT_EVERY = 500

# Duration buckets (in quarter notes)
DURATION_BUCKETS = [
    ("thirty_second",  0.125),
    ("sixteenth",      0.25),
    ("eighth",         0.5),
    ("dotted_eighth",  0.75),
    ("quarter",        1.0),
    ("dotted_quarter", 1.5),
    ("half",           2.0),
    ("dotted_half",    3.0),
    ("whole",          4.0),
    ("long",           float('inf')),
]
DURATION_LABELS = [b[0] for b in DURATION_BUCKETS]

# Velocity buckets (8 classical nuances: ppp → fff)
VELOCITY_BUCKETS = [
    ("ppp",  15),
    ("pp",   31),
    ("p",    47),
    ("mp",   63),
    ("mf",   79),
    ("f",    95),
    ("ff",   111),
    ("fff",  127),
]
VELOCITY_LABELS = [b[0] for b in VELOCITY_BUCKETS]

os.makedirs(OUTPUT_DIR, exist_ok=True)

logger.info("Sequence length  : %d", SEQUENCE_LEN)
logger.info("Min notes/file   : %d", MIN_NOTES)
logger.info("Duration buckets : %d", len(DURATION_BUCKETS))
logger.info("Velocity buckets : %d", len(VELOCITY_BUCKETS))
logger.info("Max vocab (est)  : %d tokens", 128 * len(DURATION_BUCKETS) * len(VELOCITY_BUCKETS))
logger.info("Output dir       : %s", os.path.abspath(OUTPUT_DIR))

22:01:46 | INFO     | Environment: LOCAL
22:01:46 | INFO     | Sequence length  : 64
22:01:46 | INFO     | Min notes/file   : 50
22:01:46 | INFO     | Duration buckets : 10
22:01:46 | INFO     | Velocity buckets : 8
22:01:46 | INFO     | Max vocab (est)  : 10240 tokens
22:01:46 | INFO     | Output dir       : /home/borrelle/Working/codealpha_project/task3-midi-musique-generator/output


/home/borrelle/Working/codealpha_project/task3-midi-musique-generator/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Helper functions

In [3]:
def discretize_duration(duration_ql: float) -> str:
    """Map a float duration (quarter lengths) to a bucket label."""
    for label, max_dur in DURATION_BUCKETS:
        if duration_ql <= max_dur:
            return label
    return "long"


def discretize_velocity(velocity: int) -> str:
    """
    Map a MIDI velocity (0-127) to a nuance bucket label.
    symusic always returns an int, but we clamp defensively.
    """
    velocity = max(0, min(127, int(velocity)))
    for label, max_vel in VELOCITY_BUCKETS:
        if velocity <= max_vel:
            return label
    return "fff"


def parse_midi(filepath: str) -> tuple[bool, list, str]:
    """
    Parse a single MIDI file with symusic.
    Returns (ok, events, reason)
    - events : list of (pitch, duration_bucket, velocity_bucket) tuples
    - reason : error message if ok=False

    symusic gives durations in ticks — we convert to quarter lengths
    using score.ticks_per_quarter, same unit as music21.
    """
    try:
        score = symusic.Score(filepath)
        tpq   = score.ticks_per_quarter  # ticks per quarter note
        events = []

        for track in score.tracks:
            if track.is_drum:
                continue  # skip drum tracks
            for note in track.notes:
                if note.duration <= 0:
                    continue  # skip zero-length notes
                ql  = note.duration / tpq  # convert ticks → quarter lengths
                dur = discretize_duration(ql)
                vel = discretize_velocity(note.velocity)
                events.append((note.pitch, dur, vel))

        # Sort by onset time (notes may not be ordered across tracks)
        # symusic note.time is in ticks from start
        raw_notes = []
        for track in score.tracks:
            if track.is_drum:
                continue
            for note in track.notes:
                if note.duration > 0:
                    ql  = note.duration / tpq
                    dur = discretize_duration(ql)
                    vel = discretize_velocity(note.velocity)
                    raw_notes.append((note.time, note.pitch, dur, vel))

        raw_notes.sort(key=lambda x: x[0])
        events = [(p, d, v) for _, p, d, v in raw_notes]

        if len(events) < MIN_NOTES:
            return False, [], f"too_short: {len(events)} < {MIN_NOTES}"

        return True, events, ""

    except Exception as e:
        return False, [], f"corrupt: {type(e).__name__}: {e}"


def collect_midi_files(*dirs: str, max_files: int = None) -> list[str]:
    """Recursively collect all .midi / .mid files."""
    files = []
    for d in dirs:
        files += glob.glob(os.path.join(d, "**", "*.midi"), recursive=True)
        files += glob.glob(os.path.join(d, "**", "*.mid"),  recursive=True)
    files = sorted(set(files))
    if max_files:
        files = files[:max_files]
    return files


logger.info("Helper functions defined.")

22:01:46 | INFO     | Helper functions defined.


## 4. Collect MIDI files

In [4]:
midi_files = collect_midi_files(MAESTRO_DIR, GIANTMIDI_DIR, max_files=MAX_FILES)
logger.info("Total MIDI files found: %d", len(midi_files))
logger.info("  Sample: %s", midi_files[0])
logger.info("  Sample: %s", midi_files[-1])

22:01:46 | INFO     | Total MIDI files found: 12117
22:01:46 | INFO     |   Sample: /home/borrelle/Working/codealpha_project/task3-midi-musique-generator/datasets/giantmidi-piano-unzipped-midi-v1.21-clean/a-jag-je-t-aime-juliette-oxc7fd0zn8o.mid
22:01:46 | INFO     |   Sample: /home/borrelle/Working/codealpha_project/task3-midi-musique-generator/datasets/maestro-v3.0.0/2018/MIDI-Unprocessed_Schubert7-9_MID--AUDIO_16_R2_2018_wav.midi


## 5. Speed test (10 files)

In [5]:
# Quick benchmark before full run
import time
sample = midi_files[:10]
t0 = time.time()
for f in sample:
    parse_midi(f)
elapsed = time.time() - t0
rate = len(sample) / elapsed
eta_min = len(midi_files) / rate / 60
logger.info("Speed test: %.2f files/s → ETA for %d files: ~%.0f min",
            rate, len(midi_files), eta_min)

22:01:46 | INFO     | Speed test: 99.17 files/s → ETA for 12117 files: ~2 min


## 6. Parse all MIDI files (stream to disk + checkpoint + resume)

In [6]:
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")
SKIPPED_FILE    = os.path.join(OUTPUT_DIR, "skipped_files.json")
EVENTS_FILE     = os.path.join(OUTPUT_DIR, "events_stream.pkl")

# ── Resume from checkpoint if it exists ──────────────────────────────────────
parsed_count  = 0
skipped_count = 0
skipped_files = []
processed_set = set()
total_events  = 0

if os.path.exists(CHECKPOINT_FILE):
    logger.info("Checkpoint found — resuming...")
    with open(CHECKPOINT_FILE, "r") as f:
        ckpt = json.load(f)
    processed_set = set(ckpt["processed_files"])
    parsed_count  = ckpt["parsed_count"]
    skipped_count = ckpt["skipped_count"]
    total_events  = ckpt["total_events"]
    midi_files    = [f for f in midi_files if f not in processed_set]
    logger.info("Resumed: %d done, %d remaining, %s events so far",
                len(processed_set), len(midi_files), f"{total_events:,}")

# ── Parse — stream events to disk, RAM stays flat ────────────────────────────
start_time = time.time()

with open(EVENTS_FILE, "ab") as events_fh:
    for i, filepath in enumerate(tqdm(midi_files, desc="Parsing MIDI", unit="files")):
        ok, events, reason = parse_midi(filepath)
        processed_set.add(filepath)

        if ok:
            pickle.dump(events, events_fh)   # one chunk per file
            total_events  += len(events)
            parsed_count  += 1
        else:
            skipped_count += 1
            skipped_files.append({"file": filepath, "reason": reason})

        if (i + 1) % CHECKPOINT_EVERY == 0:
            ckpt = {
                "processed_files": list(processed_set),
                "parsed_count":    parsed_count,
                "skipped_count":   skipped_count,
                "total_events":    total_events,
            }
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(ckpt, f)
            with open(SKIPPED_FILE, "w") as f:
                json.dump(skipped_files, f, indent=2)

            elapsed = time.time() - start_time
            rate    = (i + 1) / elapsed
            eta_min = (len(midi_files) - i - 1) / rate / 60
            logger.info("[%d/%d] parsed=%d skipped=%d events=%s | %.1f files/s | ETA ~%.0f min",
                        i + 1, len(midi_files), parsed_count, skipped_count,
                        f"{total_events:,}", rate, eta_min)

total_time = time.time() - start_time
logger.info("=" * 55)
logger.info("PARSE COMPLETE")
logger.info("  Files parsed  : %d", parsed_count)
logger.info("  Files skipped : %d", skipped_count)
logger.info("  Total events  : %s", f"{total_events:,}")
logger.info("  Time          : %.1f min", total_time / 60)
logger.info("=" * 55)

# ── Reload all events from disk for vocabulary building ──────────────────────
logger.info("Loading events from disk...")
all_events = []
with open(EVENTS_FILE, "rb") as f:
    while True:
        try:
            all_events.extend(pickle.load(f))
        except EOFError:
            break
logger.info("Loaded %s events.", f"{len(all_events):,}")

22:01:46 | INFO     | Checkpoint found — resuming...
22:01:46 | INFO     | Resumed: 12000 done, 117 remaining, 44,694,499 events so far


Parsing MIDI: 100%|██████████| 117/117 [00:02<00:00, 54.01files/s]

22:01:48 | INFO     | =======================================================
22:01:48 | INFO     | PARSE COMPLETE
22:01:48 | INFO     |   Files parsed  : 12109
22:01:48 | INFO     |   Files skipped : 8
22:01:48 | INFO     |   Total events  : 45,710,306
22:01:48 | INFO     |   Time          : 0.0 min
22:01:48 | INFO     | =======================================================
22:01:48 | INFO     | Loading events from disk...


22:02:48 | INFO     | Loaded 46,726,113 events.


## 7. Build vocabulary

In [7]:
unique_tokens = sorted(set(all_events))
token2idx     = {token: idx for idx, token in enumerate(unique_tokens)}
idx2token     = {idx: token for token, idx in token2idx.items()}
VOCAB_SIZE    = len(token2idx)

logger.info("Vocabulary size: %d unique (pitch, duration, velocity) tokens", VOCAB_SIZE)

# Top 10 most common tokens
counter = Counter(all_events)
print("\nTop 10 most common tokens:")
print(f"  {'pitch':<6}  {'duration':<15}  {'velocity':<6}  {'count':>10}")
print("  " + "-" * 45)
for (pitch, dur, vel), count in counter.most_common(10):
    print(f"  {pitch:<6}  {dur:<15}  {vel:<6}  {count:>10,}")

# Duration distribution
total = len(all_events)
durs = [d for _, d, _ in all_events]
dur_hist = Counter(durs)
print("\nDuration distribution:")
for label in DURATION_LABELS:
    count = dur_hist.get(label, 0)
    if count:
        bar = "█" * int(count / total * 40)
        print(f"  {label:<18} {bar} {count/total*100:.1f}%")

# Velocity distribution
vels = [v for _, _, v in all_events]
vel_hist = Counter(vels)
print("\nVelocity distribution:")
for label in VELOCITY_LABELS:
    count = vel_hist.get(label, 0)
    if count:
        bar = "█" * int(count / total * 40)
        print(f"  {label:<6} {bar} {count/total*100:.1f}%")

22:02:51 | INFO     | Vocabulary size: 6543 unique (pitch, duration, velocity) tokens

Top 10 most common tokens:
  pitch   duration         velocity       count
  ---------------------------------------------
  69      eighth           mf         110,176
  74      eighth           mf         104,446
  72      eighth           mf         103,805
  69      thirty_second    mf         103,141
  67      eighth           mf         102,564
  67      thirty_second    mf         102,016
  74      sixteenth        mf         101,619
  74      thirty_second    mf          97,011
  69      sixteenth        mf          96,821
  67      sixteenth        mf          95,804

Duration distribution:
  thirty_second      ██████ 16.4%
  sixteenth          █████ 14.8%
  eighth             ███████ 17.7%
  dotted_eighth      ████ 12.5%
  quarter            ███ 8.9%
  dotted_quarter     ████ 11.0%
  half               ██ 6.2%
  dotted_half        ██ 6.2%
  whole              █ 2.7%
  long               █ 3

## 8. Encode & build sequences

In [8]:
encoded = np.array([token2idx[e] for e in all_events], dtype=np.int32)

step = SEQUENCE_LEN // 2  # 50% overlap
X, y = [], []

for i in range(0, len(encoded) - SEQUENCE_LEN, step):
    X.append(encoded[i : i + SEQUENCE_LEN])
    y.append(encoded[i + SEQUENCE_LEN])

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

logger.info("Sequences built:")
logger.info("  X shape : %s  (n_sequences, sequence_len)", X.shape)
logger.info("  y shape : %s  (n_sequences,)", y.shape)
logger.info("  Vocab   : %d", VOCAB_SIZE)

22:03:15 | INFO     | Sequences built:
22:03:15 | INFO     |   X shape : (1460190, 64)  (n_sequences, sequence_len)
22:03:15 | INFO     |   y shape : (1460190,)  (n_sequences,)
22:03:15 | INFO     |   Vocab   : 6543


## 9. Save to disk

In [9]:
np.save(os.path.join(OUTPUT_DIR, "X.npy"), X)
np.save(os.path.join(OUTPUT_DIR, "y.npy"), y)

vocab = {
    "token2idx":       {str(k): v for k, v in token2idx.items()},
    "idx2token":       {str(k): list(v) for k, v in idx2token.items()},
    "vocab_size":      VOCAB_SIZE,
    "sequence_len":    SEQUENCE_LEN,
    "duration_labels": DURATION_LABELS,
    "velocity_labels": VELOCITY_LABELS,
    "config": {
        "min_notes":       MIN_NOTES,
        "n_files_parsed":  parsed_count,
        "n_files_skipped": skipped_count,
        "on_kaggle":       ON_KAGGLE,
    },
}
with open(os.path.join(OUTPUT_DIR, "vocab.json"), "w") as f:
    json.dump(vocab, f, indent=2)

with open(SKIPPED_FILE, "w") as f:
    json.dump(skipped_files, f, indent=2)

logger.info("Saved to %s:", os.path.abspath(OUTPUT_DIR))
for name in ["X.npy", "y.npy", "vocab.json", "skipped_files.json"]:
    path = os.path.join(OUTPUT_DIR, name)
    size_mb = os.path.getsize(path) / 1_000_000
    logger.info("  %-25s %.2f MB", name, size_mb)

22:03:15 | INFO     | Saved to /home/borrelle/Working/codealpha_project/task3-midi-musique-generator/output:
22:03:15 | INFO     |   X.npy                     373.81 MB
22:03:15 | INFO     |   y.npy                     5.84 MB
22:03:15 | INFO     |   vocab.json                0.64 MB
22:03:15 | INFO     |   skipped_files.json        0.00 MB


## 10. Sanity check

In [10]:
X_check = np.load(os.path.join(OUTPUT_DIR, "X.npy"))
y_check = np.load(os.path.join(OUTPUT_DIR, "y.npy"))
with open(os.path.join(OUTPUT_DIR, "vocab.json")) as f:
    vocab_check = json.load(f)

print(f"X shape    : {X_check.shape}")
print(f"y shape    : {y_check.shape}")
print(f"Vocab size : {vocab_check['vocab_size']}")
print(f"X max idx  : {X_check.max()} (should be < {vocab_check['vocab_size']})")
print(f"y max idx  : {y_check.max()} (should be < {vocab_check['vocab_size']})")

assert X_check.max() < vocab_check['vocab_size'], "Index out of vocab range!"
assert y_check.max() < vocab_check['vocab_size'], "Index out of vocab range!"
print("\n✓ All indices within vocabulary range")

print("\nFirst sequence (first 10 tokens):")
print(f"  {'pitch':<6}  {'duration':<15}  {'velocity'}")
print("  " + "-" * 35)
for idx in X_check[0][:10]:
    pitch, dur, vel = vocab_check['idx2token'][str(idx)]
    print(f"  {pitch:<6}  {dur:<15}  {vel}")

print("\n✓ Sanity check passed — data is ready for training.")

X shape    : (1460190, 64)
y shape    : (1460190,)
Vocab size : 6543
X max idx  : 6542 (should be < 6543)
y max idx  : 6519 (should be < 6543)

✓ All indices within vocabulary range

First sequence (first 10 tokens):
  pitch   duration         velocity
  -----------------------------------
  45      dotted_quarter   mf
  84      dotted_quarter   f
  76      dotted_eighth    mp
  48      thirty_second    mp
  52      dotted_quarter   mf
  81      dotted_quarter   mf
  76      eighth           mf
  57      dotted_quarter   mp
  84      dotted_quarter   f
  45      dotted_quarter   mf

✓ Sanity check passed — data is ready for training.
